# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a template—and practical guide—for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as an object
ds_metadata = dataset.metadata

# Print name and description
print(f"{ds_metadata.name}: {ds_metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we display the available RecordSets and associated fields, referencing all entities by their `@id`.

In [ ]:
# List all RecordSets and their @id
record_set_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in ds_metadata.recordSet]

print("Available RecordSets by @id:")
for rs_id in record_set_ids:
    print(f"  - {rs_id}")

# For each RecordSet, list its fields by @id
for rs_id in record_set_ids:
    rec_set = dataset.record_set(rs_id)
    print(f"\nFields for RecordSet {rs_id}:")
    for f in rec_set.fields:
        print(f"    - {f['@id']} : {f['name']} (type: {f.get('dataType', 'unknown')})")

## 3. Data Extraction
Load data from each available RecordSet into a DataFrame for analysis. Use the RecordSet and Field `@id`s from the overview above.

If multiple RecordSets are available, you can select them separately by their `@id`.

In [ ]:
# Extract data from all listed RecordSets
dataframes = {}

# Loop through each RecordSet by @id
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nFields for RecordSet {rs_id}:")
        print(df.columns.tolist())
        print("Sample records:")
        print(df.head())
    else:
        print(f"\nNo records found for RecordSet {rs_id}")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For this section, select a RecordSet and Field (using their `@id`) for numeric processing. If the data is tabular (e.g., has a field like 'age'), filter and normalize this field.

In [ ]:
# Example: Select first available RecordSet and a numeric field therein
selected_rs = record_set_ids[0] if record_set_ids else None

if selected_rs and selected_rs in dataframes:
    df = dataframes[selected_rs]
    # Try to find a numeric field
    # You may need to update numeric_field_id if none are found
    possible_numeric_fields = [col for col in df.columns if 'age' in col.lower() or df[col].dtype in [int, float]]
    numeric_field_id = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]
    print(f"Using numeric field: {numeric_field_id}")
    threshold = 10

    # Filtering for values greater than threshold
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalizing the field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field if available (e.g., 'sex')
    possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'anatomical' in col.lower()]
    group_field_id = possible_group_fields[0] if possible_group_fields else None
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"\nGrouped data by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No record sets or fields found for EDA.")

## 5. Visualization
Visualize distributions for numeric fields (e.g., histograms for 'age', bar plots for group counts).

_You can adjust field selections below to suit available data._

In [ ]:
import matplotlib.pyplot as plt

if selected_rs and selected_rs in dataframes:
    df = dataframes[selected_rs]
    # Use previously identified numeric and group fields
    numeric_field = numeric_field_id
    group_field = group_field_id

    # Histogram for numeric field
    if numeric_field in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field]):
        plt.figure(figsize=(6,4))
        df[numeric_field].hist(bins=10)
        plt.title(f"Distribution of {numeric_field} in {selected_rs}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()

    # Bar plot for group field
    if group_field and group_field in df.columns:
        plt.figure(figsize=(6,4))
        df[group_field].value_counts().plot(kind='bar')
        plt.title(f"Counts by {group_field} in {selected_rs}")
        plt.xlabel(group_field)
        plt.ylabel("Count")
        plt.show()
else:
    print("No data available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded FAIR^2 dataset metadata and records via the Croissant schema, reviewed RecordSets and fields by their `@id`, performed basic EDA including filtering and normalization, and visualized key distributions. Further steps may include more advanced statistical analysis, model building, or joining multiple record sets, all by referencing fields and entities via their `@id`.
